In [33]:
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("TMDB_API_KEY")

print(API_KEY)

189ae420cd9e2c08de39d4ea707eef34


In [34]:
import requests
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import time

# ==========================================
# 1. FETCH MULTILINGUAL DATA VIA API (WITH SAFEGUARDS)
# ==========================================
API_KEY = "189ae420cd9e2c08de39d4ea707eef34"  # <-- Put your actual TMDB API key here
BASE_URL = "https://api.themoviedb.org/3"

movies_list = []
languages = ['en', 'hi', 'bn']
pages_per_language = 15 

print("Starting live data fetch from TMDb API...")

for lang in languages:
    print(f"Fetching {lang} movies...")
    for page in range(1, pages_per_language + 1):
        url = f"{BASE_URL}/discover/movie?api_key={API_KEY}&with_original_language={lang}&sort_by=popularity.desc&page={page}"
        
        try:
            # Added a timeout of 10 seconds so it doesn't hang forever
            response = requests.get(url, timeout=10)
            response.raise_for_status() # Check if the request was successful
            data = response.json()
            
            if "results" in data:
                for movie in data["results"]:
                    movie_id = movie.get("id")
                    title = movie.get("title")
                    overview = movie.get("overview", "")
                    genre_ids = movie.get("genre_ids", [])
                    
                    genre_tags = " ".join([f"genre_{gid}" for gid in genre_ids])
                    
                    if title and overview:
                        movies_list.append({
                            "movie_id": movie_id,
                            "title": title,
                            "overview": overview,
                            "genres": genre_tags
                        })
                        
        except requests.exceptions.Timeout:
            print(f"⚠️ Connection timed out on page {page} for language {lang}. Skipping...")
            continue
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Network error occurred: {e}")
            break # Stop if it's a critical network breakdown
            
        time.sleep(0.2)

# Verify if we actually got data before proceeding
if len(movies_list) == 0:
    print("❌ ERROR: No movies were fetched. Check your internet connection or proxy settings.")
else:
    df = pd.DataFrame(movies_list)
    print(f"✅ Successfully fetched {len(df)} movies across English, Hindi, and Bengali!")
    
    # [Rest of your text preprocessing, Vectorization & Export code goes here...]

Starting live data fetch from TMDb API...
Fetching en movies...
⚠️ Connection timed out on page 1 for language en. Skipping...
⚠️ Connection timed out on page 2 for language en. Skipping...
⚠️ Connection timed out on page 3 for language en. Skipping...
⚠️ Connection timed out on page 4 for language en. Skipping...
⚠️ Connection timed out on page 5 for language en. Skipping...
⚠️ Connection timed out on page 6 for language en. Skipping...
⚠️ Connection timed out on page 7 for language en. Skipping...
⚠️ Connection timed out on page 8 for language en. Skipping...
⚠️ Connection timed out on page 9 for language en. Skipping...
⚠️ Connection timed out on page 10 for language en. Skipping...
⚠️ Connection timed out on page 11 for language en. Skipping...
⚠️ Connection timed out on page 12 for language en. Skipping...
⚠️ Connection timed out on page 13 for language en. Skipping...
⚠️ Connection timed out on page 14 for language en. Skipping...
⚠️ Connection timed out on page 15 for language e

KeyboardInterrupt: 

In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import time

# ==========================================
# 1. FETCH MULTILINGUAL DATA VIA API
# ==========================================
API_KEY = "189ae420cd9e2c08de39d4ea707eef34" # <-- Put your TMDB API key here
BASE_URL = "https://api.themoviedb.org/3"

movies_list = []

# Define the regions/languages you want to capture
# 'en' = Hollywood/English, 'hi' = Bollywood/Hindi, 'bn' = Bengali
languages = ['en', 'hi', 'bn']
pages_per_language = 15 # Fetches ~300 movies per language (Total ~900 movies)

print("Starting live data fetch from TMDb API...")

for lang in languages:
    print(f"Fetching {lang} movies...")
    for page in range(1, pages_per_language + 1):
        # We query the discover endpoint filtering by original language, sorted by popularity
        url = f"{BASE_URL}/discover/movie?api_key={API_KEY}&with_original_language={lang}&sort_by=popularity.desc&page={page}"
        response = requests.get(url).json()
        
        if "results" in response:
            for movie in response["results"]:
                # Safely get fields; handle missing data
                movie_id = movie.get("id")
                title = movie.get("title")
                overview = movie.get("overview", "")
                genre_ids = movie.get("genre_ids", [])
                
                # Convert genre IDs to simple strings (e.g., "Action Comedy")
                # For simplicity in this beginner setup, we use IDs as text tags
                genre_tags = " ".join([f"genre_{gid}" for gid in genre_ids])
                
                if title and overview: # Only keep if it has a title and description
                    movies_list.append({
                        "movie_id": movie_id,
                        "title": title,
                        "overview": overview,
                        "genres": genre_tags
                    })
        # Slight pause to respect API rate limits
        time.sleep(0.2)

# Convert our API results into a Pandas DataFrame
df = pd.DataFrame(movies_list)
print(f"Successfully fetched {len(df)} movies across English, Hindi, and Bengali!")

# ==========================================
# 2. CREATE THE ML "TAGS" COLUMN
# ==========================================
# Combine overview and genres into a single string for vectorization
df['tags'] = df['overview'] + " " + df['genres']
df['tags'] = df['tags'].apply(lambda x: x.lower())

# Keep only the columns needed for the recommendation engine
new_df = df[['movie_id', 'title', 'tags']]

# ==========================================
# 3. VECTORIZATION & SIMILARITY
# ==========================================
# Turn the text tags into numerical vectors
cv = CountVectorizer(max_features=3000, stop_words='english')
vector = cv.fit_transform(new_df['tags']).toarray()

# Calculate similarity matrix (scores how close movies are to each other)
similarity = cosine_similarity(vector)

# ==========================================
# 4. TEST THE LIVE RECOMMENDATION
# ==========================================
def recommend(movie_title):
    try:
        # Match case-insensitively
        movie_idx = new_df[new_df['title'].str.lower() == movie_title.lower()].index[0]
        distances = similarity[movie_idx]
        
        # Sort and get top 5 similar movies
        closest_movies = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
        
        print(f"\nRecommendations for '{movie_title}':")
        for i in closest_movies:
            print(f"- {new_df.iloc[i[0]]['title']}")
    except IndexError:
        print(f"\nMovie '{movie_title}' not found in the fetched dataset. Try another title.")

# Test it with a globally known movie or a popular regional one you expect to be fetched
recommend('3 Idiots') 

# ==========================================
# 5. EXPORT THE MODEL FOR THE BACKEND
# ==========================================
pickle.dump(new_df.to_dict(), open('movies_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print("\nModel trained and pkl files updated successfully!")

Starting live data fetch from TMDb API...
Fetching en movies...


ConnectTimeout: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/discover/movie?api_key=189ae420cd9e2c08de39d4ea707eef34&with_original_language=en&sort_by=popularity.desc&page=1 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001EF4E82D450>, 'Connection to api.themoviedb.org timed out. (connect timeout=None)'))

In [ ]:
import requests

url = "https://api.themoviedb.org/3/search/movie"

params = {
    "api_key": API_KEY,
    "query": "3 Idiots"
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [ ]:
data = response.json()

print(data)

{'page': 1, 'results': [{'adult': False, 'backdrop_path': '/8gT3UKtglLVpu0YfccwbmXZ5Eis.jpg', 'genre_ids': [18, 35], 'id': 20453, 'title': '3 Idiots', 'original_language': 'hi', 'original_title': '3 Idiots', 'overview': 'Rascal. Joker. Dreamer. Genius... You\'ve never met a college student quite like "Rancho." From the moment he arrives at India\'s most prestigious university, Rancho\'s outlandish schemes turn the campus upside down—along with the lives of his two newfound best friends. Together, they make life miserable for "Virus," the school’s uptight and heartless dean. But when Rancho catches the eye of the dean\'s daughter, Virus sets his sights on flunking out the "3 idiots" once and for all.', 'popularity': 21.7812, 'poster_path': '/66A9MqXOyVFCssoloscw79z8Tew.jpg', 'release_date': '2009-12-23', 'softcore': False, 'video': False, 'vote_average': 8.007, 'vote_count': 2748}, {'adult': False, 'backdrop_path': '/s1oSnLVAZmWyZlySy2oas250k7U.jpg', 'genre_ids': [16, 28, 35, 12], 'id':

In [ ]:
import requests

url = "https://api.themoviedb.org/3/search/movie"

params = {
    "api_key": API_KEY,
    "query": "3 Idiots"
}

try:
    response = requests.get(url, params=params, timeout=20)
    print("Status Code:", response.status_code)
    print(response.json())
except requests.exceptions.RequestException as e:
    print("Error:", e)

Error: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


In [ ]:
# is a single-line comment 

In [ ]:
print(data["results"][0]["title"])

3 Idiots


In [ ]:
print(data["results"][0]["overview"])

Rascal. Joker. Dreamer. Genius... You've never met a college student quite like "Rancho." From the moment he arrives at India's most prestigious university, Rancho's outlandish schemes turn the campus upside down—along with the lives of his two newfound best friends. Together, they make life miserable for "Virus," the school’s uptight and heartless dean. But when Rancho catches the eye of the dean's daughter, Virus sets his sights on flunking out the "3 idiots" once and for all.


In [ ]:
print(data["results"][0]["vote_average"])

8.007


In [ ]:
print(data["results"][0]["release_date"])

2009-12-23
